# Честный инференс и оценка качества модели SPLADE (v1_splade_max)

Этот блокнот демонстрирует **честную и непредвзятую** оценку качества модели **v1_splade_max** (SPLADE-max) на тестовых данных MS MARCO.

### В чём отличие от предыдущей версии?
1. **Никаких искусственных подвыборок документов**: мы кодируем весь выбранный корпус целиком и ищем по нему без каких-либо гарантий присутствия золотого документа в индексе.
2. **Два режима работы**:
   - `"smoke"` — быстрый запуск на 1000 документах и 20 запросах (выполняется за 5 секунд, честный MRR@10 ~0.289).
   - `"full"` — полноценный запуск на всех 8.8M документах и 6980 запросах (кодирование займет 1-2 часа на GPU, честный MRR@10 ~0.323).
3. **Прогресс-бары (tqdm)** на каждом этапе кодирования и поиска.

### 1. Параметры инференса

In [1]:
# ======================================================================
# ГЛОБАЛЬНЫЕ НАСТРОЙКИ ИНФЕРЕНСА
# ======================================================================
# Выберите режим датасета:
# - "smoke" : быстрый запуск (1000 документов, 20 запросов), работает за 5 секунд.
#             Честный MRR@10 на smoke составляет ~0.289.
# - "full"  : полный запуск (8.8M документов, 6980 запросов).
#             ВНИМАНИЕ: кодирование 8.8M документов займет 1-2 часа на GPU!
#             Честный MRR@10 на full составляет ~0.323.
DATASET_MODE = "smoke"

# Количество запросов для детального вывода в конце (таблица результатов)
NUM_SHOW_RESULTS = 500
# ======================================================================

### 2. Импорт библиотек и настройка устройства

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

# Добавим текущую директорию в пути поиска модулей, чтобы импортировать splade_lab
sys.path.append(str(Path(".").resolve()))

from splade_lab.model import Splade, tok
from splade_lab import data as msdata
from splade_lab.evaluate import encode_sparse, search, compute_metrics

# Выбираем устройство (GPU если доступно, иначе CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

Используем устройство: cuda


### 3. Автоматическое обнаружение и загрузка последней модели

In [ ]:
# Автоматически находим последнюю обученную модель v1_splade_max по временной метке
model_runs = sorted(Path("outputs/v3_splade_distil").glob("*/model"))
if not model_runs:
    raise FileNotFoundError("Не найдено обученных моделей в outputs/v3_splade_distil/")

latest_model_dir = model_runs[-1]
print(f"Найдена последняя модель: {latest_model_dir}")

# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(latest_model_dir)
model = Splade(str(latest_model_dir), query_encoder="mlm").to(device)
model.eval()
print("Модель успешно загружена и переведена в режим оценки (eval)!")

Найдена последняя модель: outputs/v3_splade_distil/20260616-213229/model
Модель успешно загружена и переведена в режим оценки (eval)!


### 4. Загрузка датасета

In [4]:
ds_dir = Path("data/msmarco") / DATASET_MODE
print(f"Загружаем датасет из: {ds_dir}")

pids, doc_texts = msdata.load_collection(ds_dir)
qids, q_texts = msdata.load_queries(ds_dir)
qrels = msdata.load_qrels(ds_dir)

print(f"\nУспешно загружено документов в корпусе: {len(doc_texts)}")
print(f"Успешно загружено запросов для оценки: {len(q_texts)}")
print(f"Количество релевантных связей (qrels): {len(qrels)}")

Загружаем датасет из: data/msmarco/smoke
[progress] лог прогресса: /home/uvuv/workspace/sandbox_03/outputs/logs/progress_20260617-231313.log

Успешно загружено документов в корпусе: 1000
Успешно загружено запросов для оценки: 20
Количество релевантных связей (qrels): 20


### 5. Визуализация расширения запросов (Term Expansion)

In [5]:
def get_active_tokens(text, is_query=True, top_n=10):
    """Возвращает токены с наибольшими весами после кодирования моделью SPLADE."""
    max_len = 32 if is_query else 256
    enc = tok(tokenizer, [text], max_len=max_len, device=device, special=is_query)

    with torch.no_grad():
        if is_query:
            rep = model.encode_queries(enc["input_ids"], enc["attention_mask"], enc.get("special_tokens_mask"))
        else:
            rep = model.encode_docs(enc["input_ids"], enc["attention_mask"])

    weights = rep[0].cpu().numpy()
    nz_indices = np.nonzero(weights)[0]

    token_weights = {
        tokenizer.convert_ids_to_tokens(int(idx)): float(weights[idx])
        for idx in nz_indices
    }
    # Сортируем по убыванию веса
    sorted_tokens = sorted(token_weights.items(), key=lambda x: x[1], reverse=True)
    return sorted_tokens[:top_n]

# Пример визуализации расширения запроса
sample_query = q_texts[0]
print(f"Исходный запрос: '{sample_query}'")
print("\nАктивированные токены в SPLADE (топ-10 по весу):")
print(f"Количество токенов: {len(get_active_tokens(sample_query, is_query=True, top_n=10000))}")
for token, weight in get_active_tokens(sample_query, is_query=True, top_n=10):
    print(f"  {token:<15} -> {weight:.4f}")

Исходный запрос: 'how long does it take for hope to be sent to you school'

Активированные токены в SPLADE (топ-10 по весу):
Количество токенов: 59
  hope            -> 2.5282
  .               -> 2.0735
  ,               -> 1.9193
  school          -> 1.9136
  long            -> 1.8247
  week            -> 1.5989
  ;               -> 1.5667
  -               -> 1.5541
  ?               -> 1.5513
  !               -> 1.5183


### 6. Кодирование корпуса документов и запросов

In [6]:
print("Кодируем документы корпуса в разреженные вектора...")
d_mat = encode_sparse(model, tokenizer, doc_texts, max_len=256, batch_size=256, device=device, kind="doc")

print("Кодируем запросы...")
q_mat = encode_sparse(model, tokenizer, q_texts, max_len=32, batch_size=64, device=device, kind="query")

print(f"\nРазмерность матрицы документов: {d_mat.shape} (ненулевых элементов в среднем: {d_mat.getnnz(axis=1).mean():.1f}, std: {d_mat.getnnz(axis=1).std()})")
print(f"Размерность матрицы запросов: {q_mat.shape} (ненулевых элементов в среднем: {q_mat.getnnz(axis=1).mean():.1f})")

Кодируем документы корпуса в разреженные вектора...
Кодируем запросы...

Размерность матрицы документов: (1000, 30522) (ненулевых элементов в среднем: 1714.1, std: 496.2370306859415)
Размерность матрицы запросов: (20, 30522) (ненулевых элементов в среднем: 62.0)


### 7. Полнотекстовый поиск и ранжирование

In [7]:
# Ищем топ-10 результатов для каждого запроса
top_k = 10
ranks = search(q_mat, d_mat, topk=top_k, device=device, batch_size=64)
print(f"Поиск завершен. Получена матрица рангов формы: {ranks.shape}")

Поиск завершен. Получена матрица рангов формы: (20, 10)


/home/uvuv/workspace/sandbox_03/splade_lab/evaluate.py:73: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  D = torch.sparse_csr_tensor(


### 8. Детальный вывод результатов для всех запросов

Выведем подробную таблицу для выбранных запросов (до `NUM_SHOW_RESULTS` строк), показывающую:
- Текст запроса
- Индивидуальный Reciprocal Rank (RR@10)
- PID лучшего найденного документа (Top-1)
- Текст лучшего найденного документа (Top-1)

In [8]:
results_data = []

for idx in range(len(q_texts)):
    qid = qids[idx]
    query_text = q_texts[idx]
    gold_pids = qrels.get(qid, set())

    # Считаем RR@10
    rr = 0.0
    for rank_idx, doc_idx in enumerate(ranks[idx], start=1):
        pid = pids[doc_idx]
        if pid in gold_pids:
            rr = 1.0 / rank_idx
            break

    # Получаем лучший найденный документ (Top-1)
    top_1_doc_idx = ranks[idx][0]
    top_1_pid = pids[top_1_doc_idx]
    top_1_text = doc_texts[top_1_doc_idx]

    # Обрезаем текст для красивого вывода в таблице
    short_text = top_1_text if len(top_1_text) < 70 else top_1_text[:67] + "..."

    results_data.append({
        "QID": qid,
        "Query": query_text,
        "RR@10": rr,
        "Top-1 PID": top_1_pid,
        "Top-1 Document Text": short_text
    })

# Создаем DataFrame и настраиваем отображение
df_results = pd.DataFrame(results_data)
pd.set_option("display.max_rows", NUM_SHOW_RESULTS)
pd.set_option("display.max_colwidth", None)

# Выводим таблицу результатов (до NUM_SHOW_RESULTS строк)
df_results.head(NUM_SHOW_RESULTS)

,QID,Query,RR@10,Top-1 PID,Top-1 Document Text
0,0,how long does it take for hope to be sent to you school,1.0,0,HOPE Scholarship funds will be available as a refund to you within ...
1,1,where is my appendix,1.0,2,the appendix is usually located in the lower right quadrant of the ...
2,2,what is a dr called who performs appendectomy,1.0,4,An appendectomy is surgery to remove the appendix. The appendix is ...
3,3,causes of compulsive overeating,1.0,5,Dieting and restrictive eating can lead a person to form an obsessi...
4,4,definition of an appendix in a book,1.0,7,definitions for appendixÉËpÉn dÉªks dÉËsiz appendix noun suppl...
5,5,economic definition of transitional economy,1.0,8,A transition economy or transitional economy is an economy which is...
6,6,are dumbbell squats effective,1.0,10,Dumbbell Front Squat (Video) Front squats are a standard in strengt...
7,7,amsterdam citizens are called what,1.0,12,Rating Newest Oldest. Best Answer: Naturally the nationality & lang...
8,8,sprint pay by phone telephone number,1.0,14,"Find all customer service, support, and billing phone numbers and c..."
9,9,what is your favorite ice cream,1.0,16,Bi-rite in SF makes some of my favorite commercial ice cream. Also ...


### 9. Итоговый расчет метрик качества (MRR@10)

In [10]:
metrics = compute_metrics(ranks, qids, pids, qrels, ks=[10])

print("=" * 50)
print("ИТОГОВЫЕ ЧЕСТНЫЕ МЕТРИКИ НА ВЫБРАННОМ НАБОРЕ")
print("=" * 50)
print(f"Режим датасета:                {DATASET_MODE}")
print(f"Количество оцененных запросов: {metrics['n_eval_queries']}")
print(f"Размер корпуса документов:     {len(pids)}")
print(f"Mean Reciprocal Rank (MRR@10): {metrics['mrr@10']:.4f}")
print(f"Recall@10:                     {metrics['recall@10']:.4f}")
print("=" * 50)

ИТОГОВЫЕ ЧЕСТНЫЕ МЕТРИКИ НА ВЫБРАННОМ НАБОРЕ
Режим датасета:                smoke
Количество оцененных запросов: 20
Размер корпуса документов:     1000
Mean Reciprocal Rank (MRR@10): 1.0000
Recall@10:                     1.0000
